# Bronze → Silver: Clean Raw Signals

This notebook reads raw signal data from the **bronze** layer (`bronze.raw_lifetime`, `bronze.raw_piece_info`), applies all cleaning rules, and writes validated pieces to the **silver** layer (`silver.clean_pieces`).

**Incremental**: only processes rows with timestamps newer than the latest entry in silver.

### All cleaning happens here

Silver must contain **only valid pieces**. The cleaning rules are:

1. Drop the upsetting signal (incorrectly calculated at the PLC)
2. Remove zero values (tracking failures)
3. Deduplicate timestamps per signal
4. Pivot lifetime signals into columns and join with piece identification
5. Drop rows missing piece_id or die_matrix
6. Remove extreme outliers (Q3 + 3×IQR per signal per die matrix)
7. Validate monotonic cumulative order: 2nd strike < 3rd strike < 4th strike < auxiliary press < bath
8. Compute OEE cycle time (rolling average of last 5 inter-piece intervals) and filter to 11–16s

See [03_CleaningUpData.md](../docs/03_CleaningUpData.md) for the full rationale behind each rule.

In [1]:
# TODO: implement this cell
import os
import pandas as pd
from sqlalchemy import create_engine

# Load env vars from infra/.env
for line in open("../infra/.env"):
    line = line.strip()
    if line and not line.startswith("#"):
        k, v = line.split("=", 1)
        os.environ[k] = v

engine = create_engine(
    f"postgresql://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)


## Step 1: Determine incremental boundary

Find the latest timestamp already processed in silver. Only bronze rows after this point will be processed.

In [3]:
# TODO: implement this cell
latest = pd.read_sql(
    "SELECT MAX(timestamp) AS max_ts FROM silver.clean_pieces",
    engine
)["max_ts"].iloc[0]

latest


## Step 2: Extract and filter raw signals

Read from `bronze.raw_lifetime` excluding:
- The **upsetting signal** — incorrectly calculated at the PLC, values are meaningless (range 0–6.7s with 22.8% zeros)
- **Zero values** — tracking failures where the PLC did not detect the piece at a stage

In [4]:
# TODO: implement this cell
# If silver is empty, process all data
cutoff = latest if latest is not None else "1900-01-01"
cutoff

raw_lifetime = pd.read_sql("""
SELECT timestamp, signal, value
FROM bronze.raw_lifetime
WHERE timestamp > %(cutoff)s
  AND signal <> 'forging_line.main_press.maintenance.forging_line_upsetting_lifetime_piecedata'
  AND value <> 0
ORDER BY timestamp
""", engine, params={"cutoff": cutoff})

raw_lifetime.head()


,timestamp,signal,value
0,2025-11-06 15:25:02.416000+00:00,forging_line.bath.maintenance.forging_line_lif...,70.300003
1,2025-11-06 15:25:02.416000+00:00,forging_line.general.maintenance.forging_line_...,70.300003
2,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,32.000000
3,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,38.700001
4,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,52.099998


In [ ]:
raw_lifetime.shape


## Step 3: Deduplicate timestamps

The PLC occasionally double-registers the same piece at the same timestamp. Keep only the last reading per signal.

In [5]:
# TODO: implement this cell
raw_lifetime = raw_lifetime.sort_values(["signal", "timestamp"])
dedup = raw_lifetime.drop_duplicates(subset=["signal", "timestamp"], keep="last")

dedup.head()


,timestamp,signal,value
5,2025-11-06 15:25:02.416000+00:00,forging_line.auxiliary_press.maintenance.forgi...,68.699997
11,2025-11-06 15:25:16.426000+00:00,forging_line.auxiliary_press.maintenance.forgi...,54.599998
17,2025-11-06 15:25:29.134000+00:00,forging_line.auxiliary_press.maintenance.forgi...,54.799999
23,2025-11-06 15:25:43.743000+00:00,forging_line.auxiliary_press.maintenance.forgi...,55.299999
29,2025-11-06 15:25:56.462000+00:00,forging_line.auxiliary_press.maintenance.forgi...,55.500000


## Step 4: Pivot and join

Transform the tall signal/value format into one row per piece with all cumulative times as columns. Join lifetime data with piece identification on timestamp.

forging_line.general.general.forging_line_die_matrix_piecedata
forging_line.general.general.forging_line_piece_id_piecedata


In [11]:
# 1) Pivot lifetime signals into columns
lifetime_wide = dedup.pivot_table(
    index="timestamp",
    columns="signal",
    values="value",
    aggfunc="last"
).reset_index()

# 2) Load piece info (after cutoff)
raw_piece = pd.read_sql("""
SELECT timestamp, signal, value
FROM bronze.raw_piece_info
WHERE timestamp > %(cutoff)s
ORDER BY timestamp
""", engine, params={"cutoff": cutoff})

# 3) Pivot piece info into columns
piece_wide = raw_piece.pivot_table(
    index="timestamp",
    columns="signal",
    values="value",
    aggfunc="last"
).reset_index()

# 4) Join on timestamp
joined = lifetime_wide.merge(piece_wide, on="timestamp", how="left")
joined.head()




signal,timestamp,forging_line.auxiliary_press.maintenance.forging_line_lifetime_auxiliary_press_piecedata,forging_line.bath.maintenance.forging_line_lifetime_bath_piecedata,forging_line.general.maintenance.forging_line_lifetime_piecedata,forging_line.main_press.maintenance.forging_line_lifetime_drill_piecedata,forging_line.main_press.maintenance.forging_line_lifetime_first_piecedata,forging_line.main_press.maintenance.forging_line_lifetime_second_piecedata,forging_line.general.general.forging_line_die_matrix_piecedata,forging_line.general.general.forging_line_piece_id_piecedata
0,2025-11-06 15:25:02.416000+00:00,68.699997,70.300003,70.300003,52.099998,32.000000,38.700001,5052.0,251106001720
1,2025-11-06 15:25:16.426000+00:00,54.599998,56.200001,56.200001,38.000000,17.900000,24.600000,5052.0,251106001721
2,2025-11-06 15:25:29.134000+00:00,54.799999,56.400002,56.400002,37.900002,17.900000,24.600000,5052.0,251106001722
3,2025-11-06 15:25:43.743000+00:00,55.299999,56.900002,56.900002,38.299999,18.200001,24.799999,5052.0,251106001723
4,2025-11-06 15:25:56.462000+00:00,55.500000,57.099998,57.099998,38.400002,18.400000,25.100000,5052.0,251106001724


## Step 5: Normalize and drop missing identification

Map column names to the silver schema, cast die_matrix to integer, and remove pieces missing piece_id or die_matrix.

In [16]:
# TODO: implement this cell
rename_map = {
    # lifetime signals
    "forging_line.main_press.maintenance.forging_line_lifetime_first_piecedata": "lifetime_2nd_strike_s",
    "forging_line.main_press.maintenance.forging_line_lifetime_second_piecedata": "lifetime_3rd_strike_s",
    "forging_line.main_press.maintenance.forging_line_lifetime_drill_piecedata": "lifetime_4th_strike_s",
    "forging_line.auxiliary_press.maintenance.forging_line_lifetime_auxiliary_press_piecedata": "lifetime_auxiliary_press_s",
    "forging_line.bath.maintenance.forging_line_lifetime_bath_piecedata": "lifetime_bath_s",
    "forging_line.general.maintenance.forging_line_lifetime_piecedata": "lifetime_general_s",

    # piece info signals
    "forging_line.general.general.forging_line_piece_id_piecedata": "piece_id",
    "forging_line.general.general.forging_line_die_matrix_piecedata": "die_matrix",
}

clean = joined.rename(columns=rename_map)

# drop rows without piece_id or die_matrix
clean = clean.dropna(subset=["piece_id", "die_matrix"])

# cast die_matrix to int for grouping
clean["die_matrix"] = pd.to_numeric(clean["die_matrix"], errors="coerce").astype("Int64")

clean = clean.dropna(subset=["die_matrix"])
clean["die_matrix"] = clean["die_matrix"].astype(int)


clean.head()



forging_line.auxiliary_press.maintenance.forging_line_lifetime_auxiliary_press_piecedata
forging_line.bath.maintenance.forging_line_lifetime_bath_piecedata
forging_line.general.maintenance.forging_line_lifetime_piecedata
forging_line.main_press.maintenance.forging_line_lifetime_drill_piecedata
forging_line.main_press.maintenance.forging_line_lifetime_first_piecedata
forging_line.main_press.maintenance.forging_line_lifetime_second_piecedata
forging_line.main_press.maintenance.forging_line_upsetting_lifetime_piecedata


signal,timestamp,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,lifetime_4th_strike_s,lifetime_2nd_strike_s,lifetime_3rd_strike_s,die_matrix,piece_id
0,2025-11-06 15:25:02.416000+00:00,68.699997,70.300003,70.300003,52.099998,32.000000,38.700001,5052,251106001720
1,2025-11-06 15:25:16.426000+00:00,54.599998,56.200001,56.200001,38.000000,17.900000,24.600000,5052,251106001721
2,2025-11-06 15:25:29.134000+00:00,54.799999,56.400002,56.400002,37.900002,17.900000,24.600000,5052,251106001722
3,2025-11-06 15:25:43.743000+00:00,55.299999,56.900002,56.900002,38.299999,18.200001,24.799999,5052,251106001723
4,2025-11-06 15:25:56.462000+00:00,55.500000,57.099998,57.099998,38.400002,18.400000,25.100000,5052,251106001724


## Step 6: Remove extreme outliers per die matrix

Pieces with cumulative times far outside the normal range are not slow pieces — they are **tracking failures**: stuck pieces, unclosed cycles, or machine stops that inflated the timer.

For each lifetime column, compute Q1, Q3, and IQR **per die matrix**, then remove rows where any value falls outside `[Q1 - 3×IQR, Q3 + 3×IQR]`.

In [17]:
# TODO: implement this cell
signals = [
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s"
]

def filter_outliers(group):
    mask = pd.Series(True, index=group.index)
    for s in signals:
        q1 = group[s].quantile(0.25)
        q3 = group[s].quantile(0.75)
        iqr = q3 - q1
        upper = q3 + 3 * iqr
        lower = q1 - 3 * iqr
        mask &= group[s].between(lower, upper)
    return group[mask]

before_outliers = len(clean)
clean_no_outliers = clean.groupby("die_matrix", group_keys=False).apply(filter_outliers)
after_outliers = len(clean_no_outliers)

before_outliers, after_outliers


/var/folders/l8/wb0f4qmd20n1w2y5hzv4r3xr0000gn/T/ipykernel_52857/3406982508.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  clean_no_outliers = clean.groupby("die_matrix", group_keys=False).apply(filter_outliers)


(178308, 140404)

## Step 7: Validate monotonic cumulative order

Each piece must have increasing cumulative times along the physical process:

**2nd strike < 3rd strike < 4th strike < auxiliary press < bath**

A violation means the PLC misattributed a reading or a tracking cycle overlapped. These are not valid pieces.

In [18]:
# TODO: implement this cell
df = clean_no_outliers.copy()

mask = (
    (df["lifetime_2nd_strike_s"] < df["lifetime_3rd_strike_s"]) &
    (df["lifetime_3rd_strike_s"] < df["lifetime_4th_strike_s"]) &
    (df["lifetime_4th_strike_s"] < df["lifetime_auxiliary_press_s"]) &
    (df["lifetime_auxiliary_press_s"] < df["lifetime_bath_s"])
)

before_monotonic = len(df)
clean_monotonic = df[mask]
after_monotonic = len(clean_monotonic)

before_monotonic, after_monotonic


(140404, 140404)

## Step 8: Compute OEE cycle time and filter

The OEE cycle time measures the **time between consecutive pieces** exiting the line — it is a production throughput metric. The original PLC computes it as the rolling average of the last 5 pieces at the auxiliary press.

Since the auxiliary press signal is not in our dataset, we approximate it from the **timestamp intervals** between consecutive pieces, using a rolling window of 5.

Valid OEE cycle time is **11–16 seconds**. Values outside this range indicate production stops, changeovers, or sensor errors. Pieces with invalid OEE are kept in silver (they are valid pieces) but their `oee_cycle_time_s` is set to NULL.

In [19]:
# TODO: implement this cell
df = clean_monotonic.sort_values("timestamp").copy()

# time between consecutive pieces (seconds)
df["inter_piece_s"] = df["timestamp"].diff().dt.total_seconds()

# rolling average of last 5 intervals
df["oee_cycle_time_s"] = df["inter_piece_s"].rolling(window=5, min_periods=5).mean()

# keep only valid OEE range; else set to NaN
df.loc[(df["oee_cycle_time_s"] < 11) | (df["oee_cycle_time_s"] > 16), "oee_cycle_time_s"] = pd.NA

df[["timestamp", "inter_piece_s", "oee_cycle_time_s"]].head(10)


signal,timestamp,inter_piece_s,oee_cycle_time_s
1,2025-11-06 15:25:16.426000+00:00,NaN,NaN
2,2025-11-06 15:25:29.134000+00:00,12.708,NaN
3,2025-11-06 15:25:43.743000+00:00,14.609,NaN
4,2025-11-06 15:25:56.462000+00:00,12.719,NaN
5,2025-11-06 15:26:10.462000+00:00,14.000,NaN
6,2025-11-06 15:26:23.771000+00:00,13.309,13.4690
7,2025-11-06 15:26:36.382000+00:00,12.611,13.4496
8,2025-11-06 15:26:50.095000+00:00,13.713,13.2704
12,2025-11-06 15:27:57.427000+00:00,67.332,NaN
16,2025-11-06 15:29:04.470000+00:00,67.043,NaN


## Step 9: Write to silver

Append the cleaned, validated pieces to `silver.clean_pieces`.

In [20]:
# TODO: implement this cell
silver_cols = [
    "timestamp",
    "piece_id",
    "die_matrix",
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
    "lifetime_general_s",
    "oee_cycle_time_s",
]

silver_df = df[silver_cols].copy()

silver_df.to_sql(
    "clean_pieces",
    engine,
    schema="silver",
    if_exists="append",
    index=False
)

silver_df.head()


signal,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,oee_cycle_time_s
1,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,NaN
2,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,NaN
3,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,NaN
4,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,NaN
5,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,NaN


## Cleaning Report

Summary of all cleaning decisions applied in this run, with counts and justifications.

In [21]:
report = {
    "raw_lifetime_rows": len(raw_lifetime),
    "after_dedup": len(dedup),
    "after_join": len(joined),
    "after_drop_missing_id": len(clean),
    "after_outliers": len(clean_no_outliers),
    "after_monotonic": len(clean_monotonic),
    "final_silver_rows_written": len(silver_df),
}

pd.DataFrame.from_dict(report, orient="index", columns=["count"])



,count
raw_lifetime_rows,1041278
after_dedup,1041272
after_join,183452
after_drop_missing_id,178308
after_outliers,140404
after_monotonic,140404
final_silver_rows_written,140404
